# conv-output-shape — faded example 3: Complete the per-layer update in a conv-stack shape trace

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-output-shape`. The last cell reports your progress on the `CNN: Conv output shape` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Conv output shape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-output-shape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-output-shape"
DD_SUBTOPIC = "CNN: Conv output shape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To trace spatial size through stacked conv layers, repeatedly apply `L = (L + 2P - K)//S + 1`, feeding each layer's output size as the next layer's input. There is no closed form for the whole stack — only iterative application.

## Faded exercise 3

### Complete `trace_conv_stack`

`layers` is a list of `(K, S, P)` tuples applied in order to a square feature map of side `L`. The loop is set up for you. **You must fill in the body of the loop: update `L` to this layer's output spatial size** using the conv formula. The test chains real `nn.Conv2d` layers and compares the final side length.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def trace_conv_stack(L, layers):
    for (K, S, P) in layers:
        L = (L + 2 * P - K) // S + 1
    return L


def _test():
    layers = [(3, 1, 1), (3, 2, 1), (5, 2, 0)]
    pred = trace_conv_stack(64, layers)
    net = t.nn.Sequential(
        t.nn.Conv2d(3, 8, 3, stride=1, padding=1),
        t.nn.Conv2d(8, 16, 3, stride=2, padding=1),
        t.nn.Conv2d(16, 32, 5, stride=2, padding=0),
    )
    actual = net(t.zeros(1, 3, 64, 64)).shape[-1]
    assert pred == actual, (pred, actual)

    layers2 = [(7, 2, 3), (3, 1, 0), (3, 2, 1)]
    pred2 = trace_conv_stack(100, layers2)
    net2 = t.nn.Sequential(
        t.nn.Conv2d(1, 4, 7, stride=2, padding=3),
        t.nn.Conv2d(4, 4, 3, stride=1, padding=0),
        t.nn.Conv2d(4, 4, 3, stride=2, padding=1),
    )
    actual2 = net2(t.zeros(1, 1, 100, 100)).shape[-1]
    assert pred2 == actual2, (pred2, actual2)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def trace_conv_stack(L, layers):
    for (K, S, P) in layers:
        L = (L + 2 * P - K) // S + 1
    return L
```
</details>